In [1]:
# 1. 读取数据
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"数据集长度（字符数）: {len(text)}")

# 2. 看看前 100 个字符
print("--- 前 100 个字符 ---")
print(text[:100])

# 3. 这里是模型要处理的所有字符（词汇表）
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"词汇表: {''.join(chars)}")
print(f"词汇表大小: {vocab_size}")

数据集长度（字符数）: 1115394
--- 前 100 个字符 ---
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You
词汇表: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
词汇表大小: 65


In [2]:
stoi= { ch:i for i,ch in enumerate(chars) } 
itos= { i:ch for i,ch in enumerate(chars) } 

def encode(s):
    return [stoi[c] for c in s]

decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string
# 4. 测试编码和解码
print("--- 测试编码和解码 ---")
print(encode("hello world"))
print(len(encode("hello world")))
print(decode(encode("hello world")))
print(encode(" "))

--- 测试编码和解码 ---
[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
11
hello world
[1]


In [3]:
itos

{0: '\n',
 1: ' ',
 2: '!',
 3: '$',
 4: '&',
 5: "'",
 6: ',',
 7: '-',
 8: '.',
 9: '3',
 10: ':',
 11: ';',
 12: '?',
 13: 'A',
 14: 'B',
 15: 'C',
 16: 'D',
 17: 'E',
 18: 'F',
 19: 'G',
 20: 'H',
 21: 'I',
 22: 'J',
 23: 'K',
 24: 'L',
 25: 'M',
 26: 'N',
 27: 'O',
 28: 'P',
 29: 'Q',
 30: 'R',
 31: 'S',
 32: 'T',
 33: 'U',
 34: 'V',
 35: 'W',
 36: 'X',
 37: 'Y',
 38: 'Z',
 39: 'a',
 40: 'b',
 41: 'c',
 42: 'd',
 43: 'e',
 44: 'f',
 45: 'g',
 46: 'h',
 47: 'i',
 48: 'j',
 49: 'k',
 50: 'l',
 51: 'm',
 52: 'n',
 53: 'o',
 54: 'p',
 55: 'q',
 56: 'r',
 57: 's',
 58: 't',
 59: 'u',
 60: 'v',
 61: 'w',
 62: 'x',
 63: 'y',
 64: 'z'}

In [4]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"Is CUDA available? {torch.cuda.is_available()}")

# 既然是轻薄本没有GPU，我们可以确认一下当前使用的设备
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Current device: {device}")

PyTorch version: 2.5.1
Is CUDA available? False
Current device: cpu


In [5]:
data=torch.tensor(encode(text), dtype=torch.long)
print(f"数据的形状: {data.shape}, 数据类型: {data.dtype}")
print(data[:1000])  # 打印前1000个编码后的数据

数据的形状: torch.Size([1115394]), 数据类型: torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 

In [6]:
#接下来把数据分开，分成训练集和验证集
n = int(0.9*len(data)) # 前90%作为训练集
train_data = data[:n]
val_data = data[n:]

In [7]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [8]:
x=train_data[:block_size]
y=train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"当 t={t} 时，输入的上下文是 {context}，目标是 {target}")

当 t=0 时，输入的上下文是 tensor([18])，目标是 47
当 t=1 时，输入的上下文是 tensor([18, 47])，目标是 56
当 t=2 时，输入的上下文是 tensor([18, 47, 56])，目标是 57
当 t=3 时，输入的上下文是 tensor([18, 47, 56, 57])，目标是 58
当 t=4 时，输入的上下文是 tensor([18, 47, 56, 57, 58])，目标是 1
当 t=5 时，输入的上下文是 tensor([18, 47, 56, 57, 58,  1])，目标是 15
当 t=6 时，输入的上下文是 tensor([18, 47, 56, 57, 58,  1, 15])，目标是 47
当 t=7 时，输入的上下文是 tensor([18, 47, 56, 57, 58,  1, 15, 47])，目标是 58


In [9]:
import torch

torch.manual_seed(1337)
batch_size = 4  # how many independent sequences will we process in parallel?
block_size = 8  # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data #选取数据集
    ix = torch.randint(len(data) - block_size, (batch_size,)) #随机选取batch_size个起始位置
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)
print('-----')

for b in range(batch_size):  # batch dimension
    for t in range(block_size):  # time dimension
        context = xb[b, :t+1] #这是取从第0个位置到第t个位置的所有输入
        target = yb[b, t] #这是第t个位置的目标
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
-----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 5

In [10]:
print(xb)

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [11]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # 创建一个嵌入表 (Embedding Table)
        # 行数 = vocab_size (每个可能的输入字符对应一行)
        # 列数 = vocab_size (每个可能的输出字符对应一列，代表 logits)
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx 和 targets 都是 (B,T) 的整数张量
        logits = self.token_embedding_table(idx) # (B,T,C)
        
        if targets is None:
            loss = None
        else:
            # --- 修复代码开始 ---
            B, T, C = logits.shape
            # 1. 重新变形 logits，变成 (Batch*Time, Channel) 即 (32, 65)
            logits = logits.view(B*T, C)
            # 2. 重新变形 targets，变成 (Batch*Time) 即 (32)
            targets = targets.view(B*T)
            # 3. 现在形状符合标准 (N, C) 和 (N) 了
            loss = F.cross_entropy(logits, targets)
            # --- 修复代码结束 ---
        
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx: 这里的输入 idx 就是我们常说的 "Prompt"（提示词）。
        # 它的形状是 (B, T)，比如 (1, 1) 代表我们给模型一个初始字符（例如回车符 '0'）。
        
        for _ in range(max_new_tokens): # 循环 100 次，生成 100 个新字符
            
            # 1. 预测 (Get Predictions)
            # 把当前的整个序列 idx 扔给模型
            logits, loss = self(idx) 
            
            # 2. 只看最后一步 (Focus on last time step)
            # 模型虽然输出了对 idx 里每一个字符的预测，但我们只关心最后一个！
            # 因为我们要预测的是“未来”的那个新字符。
            logits = logits[:, -1, :] # 形状变成 (B, C)
            
            # 3. 计算概率 (Softmax)
            # 把得分变成概率分布。比如 ['a': 0.1, 'b': 0.8, 'c': 0.1]
            probs = F.softmax(logits, dim=-1) # (B, C)
            
            # 4. 随机抽样 (Sample)
            # 依据概率随机选这一个字。
            # 这里用 multinomial 而不是 argmax（直接选概率最大的），是为了保证生成的文本具有多样性。
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            
            # 5. 拼接 (Concatenate)
            # 把新预测出来的字符 idx_next 拼到原来的 idx 后面。
            # 此时 idx 长度 +1，并在下一次循环中作为输入。
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
            
        return idx
    
m = BigramLanguageModel(vocab_size)
logits,loss = m(xb, yb)
print('这是xb', xb)
print('这是yb', yb)
print('这是logits', logits)
print(logits.shape)
print(loss.shape)  # loss 是一个标量，所以它没有形状
print(loss)



这是xb tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
这是yb tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
这是logits tensor([[-1.5101, -0.0948,  1.0927,  ..., -0.6126, -0.6597,  0.7624],
        [ 0.3323, -0.0872, -0.7470,  ..., -0.6716, -0.9572, -0.9594],
        [ 0.2475, -0.6349, -1.2909,  ...,  1.3064, -0.2256, -1.8305],
        ...,
        [-2.1910, -0.7574,  1.9656,  ..., -0.3580,  0.8585, -0.6161],
        [ 0.5978, -0.0514, -0.0646,  ..., -1.4649, -2.0555,  1.8275],
        [-0.6787,  0.8662, -1.6433,  ...,  2.3671, -0.7775, -0.2586]],
       grad_fn=<ViewBackward0>)
torch.Size([32, 65])
torch.Size([])
tensor(4.8786, grad_fn=<NllLossBackward0>)


In [12]:
idx=torch.zeros((1, 1), dtype=torch.long)
idy=m.generate(idx = idx, max_new_tokens=10)
print(idy)
print(idy.shape)
print('------------')
now=idy[0].tolist()
print(now)
print(len(now))
print('------------')
now1=decode(now)
print(now1)
print(len(now1))
print('------------')

print(decode(m.generate(idx = idx, max_new_tokens=100)[0].tolist()))


tensor([[ 0, 31, 56, 12, 55, 28,  7, 29, 35, 49, 58]])
torch.Size([1, 11])
------------
[0, 31, 56, 12, 55, 28, 7, 29, 35, 49, 58]
11
------------

Sr?qP-QWkt
11
------------

XoL&jLDJgOLVz'RIoDqHdhsV&vLLxatjscMpwLERSPyao.qfzs$Ys$zF-w,;eEkzxjgCKFChs!iWW.ObzDnxA Ms$3!dcbf?pGXe


In [13]:
# 获取参数总数
total_params = sum(p.numel() for p in m.parameters())
print(f"总参数量: {total_params}")

# 查看参数形状
for param in m.parameters():
    print(param.shape)

总参数量: 4225
torch.Size([65, 65])


In [14]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
print(optimizer)

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.01
)


In [15]:
batch_size = 32
for steps in range(10000):
    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
  
print(loss.item())

2.45886492729187


In [16]:
print(decode(m.generate(idx = idx, max_new_tokens=100)[0].tolist()))


SPabe pave pirance
Rie hicomyonthar's
Plinseard ith henoure wounonthioneir thondy, y heltieiengerofo


# 下面开始是self-attention

In [17]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x=torch.randn(B, T, C)
print(x.shape)

torch.Size([4, 8, 2])


In [18]:
xbow=torch.zeros(B, T, C)
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t, C)
        xbow[b, t] = torch.mean(xprev, 0) # (C,)


In [19]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(dim=1, keepdim=True)
xbow2 = wei @ x  #(T,T) @ (B,T,C) -> (B,T,C) #这里用到了广播机制
torch.allclose(xbow, xbow2)

False

In [20]:
tril=torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=1)  # 注意这里应该是 dim=1，因为我们要对每一行进行 softmax
xbow3 = wei @ x
torch.allclose(xbow2, xbow3)

True

In [21]:
# ... 之前的代码 ...
diff = torch.abs(xbow - xbow2)
print("最大差异:", diff.max().item())
print("平均差异:", diff.mean().item())
# ... 后续代码 ...

最大差异: 3.236345946788788e-08
平均差异: 7.214111974462867e-09


In [22]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [23]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

In [24]:
xbow2[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

In [25]:
torch.tril(torch.ones(3, 3))

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

In [26]:
torch.manual_seed(42)
a=torch.tril(torch.ones(3,3))
a = a / torch.sum(a, dim=1, keepdim=True)
b=torch.randint(0, 10, (3,2)).float()
c=a @ b
print('a')
print(a)
print('b')
print(b)
print('c')
print(c)

a
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [27]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x=torch.randn(B, T, C)

#只有一个头的自注意力机制
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, head_size)
q = query(x) # (B, T, head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5 # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)

tril=torch.tril(torch.ones(T, T))
# wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=1)  # 注意这里应该是 dim=1，因为我们要对每一行进行 softmax

v=value(x) # (B, T, head_size)
out = wei @ v # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
out.shape

torch.Size([4, 8, 16])

上面这个代码块完整的实现了一个简单的自注意力机制（Self-Attention Mechanism）。下面重点解释经典公式：
1.为什么要除head_size**-0.5？
因为在计算注意力权重时，我们会计算 query 和 key 的点积。随着 head_size 的增加，这个点积的值可能会变得非常大，导致 softmax 函数的梯度变得非常小（梯度消失问题）。为了避免这个问题，我们通常会将点积结果除以 head_size 的平方根（head_size**0.5），这样可以保持数值稳定性，使得 softmax 的输出不会过于极端，从而有助于模型的训练。
比如相同的一组数据，如果直接进入softmax会输出一组概率a,如果把这组数据全部扩大5倍，再经过softmax会输出一组概率b,我们会得到b要比a尖锐，也就是说太绝对里，不利于训练

In [ ]:
class BatchNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps

        self.gamma = torch.ones(dim) # 缩放参数，初始为 1
        self.beta = torch.zeros(dim) # 平移参数，初始为 0
    def __call__(self, x):
        xmean = x.mean(1,keepdim=True) # 计算当前批次的均值
        xvar = x.var(1,keepdim=True) # 计算当前批次的方差
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # 标准化
        self.out = self.gamma * xhat + self.beta # 缩放和平移
        return self.out
    def parameters(self):
        return [self.gamma, self.beta]
    
torch.manual_seed(1337)
module = BatchNorm1d(100)
x = torch.randn(32,100)
x = module(x)
print(x.shape)

torch.Size([32, 100])


In [29]:
x[:,0].mean(), x[:,0].std()

(tensor(7.4506e-09), tensor(1.0000))

In [32]:
x[0,:].mean(), x[0,:].std()

(tensor(-9.5367e-09), tensor(1.0000))